In [0]:
from pyspark.sql import functions as F
from datetime import datetime

files = dbutils.fs.ls("abfss://bronzeshippingdata@shippingdata.dfs.core.windows.net/bronze")

folder = [f for f in files if f.isDir()]

lates_folder = max(folder,key=lambda f: datetime.strptime(f.name.strip("/"),"%Y-%m-%d"))

latest_folder= lates_folder.path

print('lates_folder_path',latest_folder)

files = dbutils.fs.ls(latest_folder)

for f in files:
    print(f.path)

max_time = max(f.modificationTime for f in files)

latest_file = [f for f in files if f.modificationTime == max_time]

print('latest_file',latest_file)

csv_files = [f.path for f in latest_file if f.path.endswith(".csv")]

df_csv = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(csv_files) \
    .select(
        "*",
        F.col("_metadata.file_path").alias("file_path"),
        F.col("_metadata.file_name").alias("file_name")
    )

display(df_csv)


In [0]:
correct_schema = df_csv \
    .withColumn('shipment_id', F.col('shipment_id').cast('varchar(10)')) \
    .withColumn('order_id', F.col('order_id').cast('varchar(10)'))\
    .withColumn('customer_id', F.col('customer_id').cast('varchar(10)')) \
    .withColumn('carrier_name', F.col('carrier_name').cast('string')) \
    .withColumn('shipment_type', F.col('shipment_type').cast('string')) \
    .withColumn('origin_hub', F.col('origin_hub').cast('string')) \
    .withColumn('destination_hub', F.col('destination_hub').cast('string')) \
    .withColumn('distance_km', F.col('distance_km').cast('float')) \
    .withColumn('zone', F.col('zone').cast('string')) \
    .withColumn('weight_kg', F.col('weight_kg').cast('float')) \
    .withColumn('booking_timestamp', F.col('booking_timestamp').cast('timestamp')) \
    .withColumn('sla_hours', F.col('sla_hours').cast('int')) \
    .withColumn('expected_delivery_ts', F.col('expected_delivery_ts').cast('timestamp')) \
    .withColumn('file_path', F.col('file_path').cast('string')) \
    .withColumn('file_name', F.col('file_name').cast('string'))

display(correct_schema)

In [0]:
from pyspark.sql.functions import col

files = dbutils.fs.ls("abfss://bronzeshippingdata@shippingdata.dfs.core.windows.net/bronze/2026-04-20/hub_master_2026-04-20_05-54-51.csv")

csv_files = [f.path for f in files if f.path.endswith(".csv")]

df_csv = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(csv_files) \
    .select(
        "*",
        col("_metadata.file_path").alias("file_path"),
        col("_metadata.file_name").alias("file_name")
    )

display(df_csv)

In [0]:
shippment_csv = df_csv.withColumn('primary_hub',F.col('hub_name'))\
    .select("primary_hub","hub_id","expected_dwell_hrs")

display(shippment_csv)

In [0]:
files_json = dbutils.fs.ls("abfss://bronzeshippingdata@shippingdata.dfs.core.windows.net/bronze/2026-04-20/checkpoint_scan_events_2026-04-20_05-54-51.json")

json_files = [f.path for f in files_json if f.path.endswith(".json")]

df_json = spark.read.format("json") \
    .option("multiline", "true") \
    .load(json_files) \
    .select(
        "*",
        col("_metadata.file_path").alias("file_path"),
        col("_metadata.file_name").alias("file_name")
    )

display(df_json)

In [0]:
shippemt_final = correct_schema.join(df_json, on='shipment_id',how="inner" )

shippemt_columns= shippemt_final.select('shipment_id','order_id','customer_id','carrier_name','shipment_type','origin_hub','destination_hub','distance_km','zone','weight_kg','booking_timestamp','sla_hours','expected_delivery_ts','delay_profile','event_id','hub_name','scan_status','scan_timestamp','scan_type','hub_id')

display(shippemt_columns)

In [0]:
# Total time shippemnt is spent in the hub

from pyspark.sql.window import Window

windowSpec = Window.partitionBy('shipment_id').orderBy(F.col('scan_timestamp').asc())

shippment_time = shippemt_columns.withColumn('next_scan_type', F.lead('scan_type',1).over(windowSpec))\
    .withColumn('next_scan_timestamp',F.lead('scan_timestamp',1).over(windowSpec)) \
    .withColumn('duration_hours',F.unix_timestamp(F.col('next_scan_timestamp')) - F.unix_timestamp(F.col('scan_timestamp')))\
    .withColumn('date', F.to_date('scan_timestamp'))\
    .withColumn('Total_hours_delay_hours_per_id',F.col('duration_hours')/3600)

df = shippment_time.filter(
    ((F.col("scan_type") == "PICKED_UP") & (F.col("next_scan_type") == "DEPARTED")) |
    ((F.col("scan_type") == "ARRIVED") & (F.col("next_scan_type") == "DEPARTED")) |
    ((F.col("scan_type") == "ARRIVED") & (F.col("next_scan_type") == "DELIVERED"))
)

display(df)



In [0]:
ship = df.join(shippment_csv, on='hub_id', how='left')
shipes = ship.withColumn(
    'expected_dwell_hrs',
   F.coalesce(F.col('expected_dwell_hrs'),F.lit(0))
)

display(shipes)

In [0]:

shippemt_per_order = shipes.withColumn(
    'delay_hrs',
    (F.col('Total_hours_delay_hours_per_id') - F.col('expected_dwell_hrs'))
).withColumn(
    'delay_category',
    F.when(F.col('delay_hrs') <=1, 'normal')
    .when(F.col('delay_hrs') <= 4, 'minor_delay')
    .otherwise('major_delay')
)


display(shippemt_per_order)

In [0]:
shippment_final = shippemt_per_order.filter( 
        ((F.col("scan_type") == "PICKED_UP") & (F.col("next_scan_type") == "DEPARTED")) |
        ((F.col("scan_type") == "ARRIVED") & (F.col("next_scan_type")=="DEPARTED")) |
        ((F.col("scan_type") == "ARRIVED") & (F.col("next_scan_type") == "DELIVERED")
)
)

final_result = shippment_final.groupBy('hub_name','date')\
    .agg(
         F.first('hub_id').alias('hub_id'),
         F.count('shipment_id').alias('Total_Shippment_of_day'),
         F.count(F.when((F.col('delay_category')== 'minor_delay') | (F.col('delay_category')== 'major_delay'), True)).alias('Total_Delay_Shippment_of_day'),
         F.sum(F.when(F.col('delay_hrs') > 1, F.col('delay_hrs')).otherwise(0)).alias('Total_Delay_Hours_of_day'),

    )
display(final_result)


In [0]:
final_result_avg = final_result.withColumn(
    'Avg_Delay_Hours_of_day',
    F.when(
        F.col('Total_Delay_Shippment_of_day') !=0,
        F.round(
            F.col('Total_Delay_Hours_of_day') / F.col('Total_Delay_Shippment_of_day'),
            2
        )
    ).otherwise(0)
)

display(final_result_avg)